# DINOv3 ConvNeXt pyramid decoder — crop emergence counter (training)

Frozen DINOv3 ConvNeXt backbone -> pyramid decoder -> stride-4 peak heatmap
-> local-max decode -> **points + count**. All reusable code is the
`cropcounter` package (`pip install -e .` from the repo root):

- `cropcounter.crop_dataset` — CVAT / COCO-keypoints / Datumaro parsers, the
  train/val split loader, the native-resolution tile dataset
- `cropcounter.dinov3_pyramid` — frozen backbone + pyramid decoder
- `cropcounter.heatmap` — Gaussian target rendering + local-max peak decoding
- `cropcounter.losses`, `cropcounter.metrics`, `cropcounter.train`

Runs against the bundled anonymised sample set in `../examples/data` (20
images) so it works out of the box with no private data. Point it at your
own dataset by changing `cfg.data_root` below — see `examples/README.md`
for the expected `{train,val}/{annotations.xml, images/}` layout.

Run top to bottom from the `notebooks/` directory; training artifacts land
in `runs/<run_name>/` (`best.pt`, `last.pt`, `history.json`, `curves.png`),
gitignored.

In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

from cropcounter import (
    CropTileDataset, TrainConfig, collate_val, decode_classes, load_checkpoint,
    load_splits, match_points, render_class_targets, resolve_device, sweep_tau, train,
)
from cropcounter.dinov3_pyramid import IMAGENET_MEAN, IMAGENET_STD

device = resolve_device()
print(torch.__version__, device)

# One colour per class for the overlays below (class index -> colour).
class_colour = plt.get_cmap("tab10")

## Config

Load the recorded hyperparameters for the 13-epoch reference run, then repoint the data/weights/output paths at this repo's bundled sample set — the notebook is self-contained and doesn't need the private full dataset to run.

In [ ]:
cfg = TrainConfig.from_json(Path("../examples/demo/config_13ep.json"))
cfg.data_root = Path("../examples/demo/data")
cfg.weights_dir = Path("../weights")
cfg.out_dir = Path("runs")

assert cfg.data_root.is_dir(), f"missing {cfg.data_root.resolve()} — run from notebooks/"
cfg

## Data

In [ ]:
# Pre-split on disk: data_root/{train,val}/{annotations.xml, images/}.
# cfg.labels is the label filter derived from cfg.classes (None = keep all).
train_recs, val_recs = load_splits(cfg.data_root, fmt=cfg.annotation_format, labels=cfg.labels)

print(f"total: {len(train_recs) + len(val_recs)} images, "
      f"{sum(len(r.points) for r in train_recs + val_recs)} points")
print(f"train: {len(train_recs)} images ({sum(len(r.points) for r in train_recs)} pts)")
print(f"val:   {len(val_recs)} images ({sum(len(r.points) for r in val_recs)} pts)")

# Points per output class (channel), through the label -> channel map.
print(f"classes ({cfg.n_classes}): {cfg.classes}")
for split_name, recs in (("train", train_recs), ("val", val_recs)):
    per_class = {name: 0 for name in cfg.class_names}
    for r in recs:
        for pt in r.points:
            idx = 0 if cfg.class_map is None else cfg.class_map.get(pt.label)
            if idx is not None:
                per_class[cfg.class_names[idx]] += 1
    print(f"  {split_name}: " + ", ".join(f"{k} {v}" for k, v in per_class.items()))

## Sanity checks

Cheap checks that run before spending any compute on training.

In [ ]:
# One augmented training tile, its Gaussian target, and the points decoded
# straight back off the target. Train-mode CropTileDataset returns a plain
# (image, target, n_points) tuple, not the dict val mode uses below.
sanity_ds = CropTileDataset(
    train_recs, cfg.train_images_dir, train=True, tile=cfg.tile,
    output_stride=cfg.output_stride, sigma=cfg.sigma,
    tiles_per_image=cfg.tiles_per_image, scale_jitter=cfg.scale_jitter,
    exclude_label_statuses=cfg.exclude_label_statuses,
    class_map=cfg.class_map, n_classes=cfg.n_classes,
)
img_t, target, n_pts = sanity_ds[np.random.randint(len(sanity_ds))]   # target: (C, H, W)
img = img_t.permute(1, 2, 0).numpy() * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
pts, _, cls = decode_classes(
    target, cfg.class_names, tau=0.5, k=cfg.k, nms_radius=cfg.nms_radius,
    output_stride=cfg.output_stride,
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
axes[0].imshow(img.clip(0, 1))
axes[0].set_title(f"augmented tile ({n_pts} points)")
axes[1].imshow(target.max(0).values.numpy(), cmap="hot")   # max over class channels
axes[1].set_title(f"target (stride {cfg.output_stride}, sigma {cfg.sigma}, "
                  f"{cfg.n_classes} class(es))")
axes[2].imshow(img.clip(0, 1))
for c, name in enumerate(cfg.class_names):
    m = cls == c
    axes[2].scatter(pts[m, 0], pts[m, 1], s=45, facecolors="none",
                    edgecolors=[class_colour(c)], label=f"{name}: {int(m.sum())}")
axes[2].legend(loc="upper right", fontsize=8)
axes[2].set_title(f"decoded back from target: {len(pts)}")
for ax in axes:
    ax.axis("off")

In [ ]:
# Target -> decode round trip over the val split: checks sigma / k / stride
# geometry is self-consistent before any training.
# Uses the val-mode dataset so the label -> channel mapping and QC filtering
# are exactly what training sees; reported per class, so a class-specific
# sigma / k problem shows up on its own line.
roundtrip_ds = CropTileDataset(
    val_recs, cfg.val_images_dir, train=False, output_stride=cfg.output_stride,
    sigma=cfg.sigma, exclude_label_statuses=cfg.exclude_label_statuses,
    class_map=cfg.class_map, n_classes=cfg.n_classes,
)
tot = {name: {"gt": 0, "dec": 0, "tp": 0} for name in cfg.class_names}
for i, r in enumerate(val_recs):
    gt, gt_ids = roundtrip_ds.points_px[i], roundtrip_ds.class_ids[i]
    out_h = (r.height + 31) // 32 * 32 // cfg.output_stride
    out_w = (r.width + 31) // 32 * 32 // cfg.output_stride
    t = render_class_targets(gt / cfg.output_stride, gt_ids, cfg.n_classes,
                             (out_h, out_w), cfg.sigma)
    dec, _, dec_ids = decode_classes(
        t, cfg.class_names, tau=0.5, k=cfg.k, nms_radius=cfg.nms_radius,
        output_stride=cfg.output_stride,
    )
    for c, name in enumerate(cfg.class_names):
        tp, _, _ = match_points(dec[dec_ids == c], gt[gt_ids == c],
                                radius_px=2.5 * cfg.output_stride)
        tot[name]["gt"] += int((gt_ids == c).sum())
        tot[name]["dec"] += int((dec_ids == c).sum())
        tot[name]["tp"] += tp

for name, s in tot.items():
    line = f"{name}: GT {s['gt']} -> decoded {s['dec']}"
    if s["gt"]:
        line += (f" | recovered {100 * s['tp'] / s['gt']:.2f}%"
                 f" | merge loss {100 * (s['gt'] - s['dec']) / s['gt']:.2f}%")
    print(line)

## Train

AdamW on the decoder only (backbone frozen), warmup + cosine, bf16 autocast on CUDA. Per-epoch whole-image validation at `cfg.tau`; best checkpoint = lowest val loss.

In [ ]:
model, history, run_name, best_epoch = train(cfg)

#### Check

In [ ]:
# Load the best checkpoint from the run above.
run_dir = cfg.out_dir / run_name
print(f"loading {run_dir / 'best.pt'}")
model, _ = load_checkpoint(run_dir / "best.pt", device)
model.eval()

In [ ]:
# Alternatively, the last epoch's checkpoint instead of the best-val one:
# model, _ = load_checkpoint(run_dir / "last.pt", device)
# model.eval()

In [ ]:
tau_thresh = 0.35  # cfg.tau; one value for every class, or {class_name: tau}
k_kernel = cfg.k
rng_val = 7

val_ds = CropTileDataset(
    val_recs, cfg.val_images_dir, train=False, output_stride=cfg.output_stride,
    sigma=cfg.sigma, exclude_label_statuses=cfg.exclude_label_statuses,
    class_map=cfg.class_map, n_classes=cfg.n_classes,
)
n_show = min(3, len(val_ds))
picks = np.random.default_rng(rng_val).choice(len(val_ds), size=n_show, replace=False)

fig, axes = plt.subplots(n_show, 2, figsize=(15, 20 * n_show / 3), squeeze=False)
for row, idx in enumerate(picks):
    item = val_ds[int(idx)]
    with torch.no_grad(), torch.autocast(device.type, torch.bfloat16, enabled=device.type == "cuda"):
        logits = model(item["image"].unsqueeze(0).to(device))
    prob = torch.sigmoid(logits.float()).cpu()                     # (1, C, h, w)
    pred, _, pred_ids = decode_classes(
        prob, cfg.class_names, tau=tau_thresh, k=k_kernel, nms_radius=cfg.nms_radius,
        output_stride=cfg.output_stride,
    )
    img = item["image"].permute(1, 2, 0).numpy() * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    gt, gt_ids = item["points"], item["class_ids"]
    axes[row, 0].imshow(img.clip(0, 1))
    for c, name in enumerate(cfg.class_names):   # GT filled, predictions as rings
        g, pm = gt_ids == c, pred_ids == c
        axes[row, 0].scatter(gt[g, 0], gt[g, 1], s=25, color=class_colour(c), alpha=0.8,
                             label=f"GT {name} {int(g.sum())}")
        axes[row, 0].scatter(pred[pm, 0], pred[pm, 1], s=60, facecolors="none",
                             edgecolors=[class_colour(c)], label=f"pred {name} {int(pm.sum())}")
    axes[row, 0].legend(loc="upper right", fontsize=8)
    axes[row, 0].set_title(item["name"][:70], fontsize=9)
    axes[row, 1].imshow(prob[0].max(0).values, cmap="hot", vmin=0, vmax=1)
    axes[row, 1].set_title("predicted heatmap" + (" (max over classes)" if cfg.n_classes > 1 else ""))
    for ax in axes[row]:
        ax.axis("off")

In [ ]:
def calibrate_tau(cfg, model, val_ds):
    """Calibrate the decode threshold: one forward pass over val, decoded
    at every tau. Picks the tau minimising count MAE (the macro mean over
    classes) and -- because channels decode independently -- the best tau
    of every class on its own. That per-class dict can go straight into
    ``cfg.tau`` for a multiclass model."""
    val_loader = DataLoader(val_ds, batch_size=1, collate_fn=collate_val)
    taus = np.round(np.arange(0.05, 0.75, 0.05), 2)
    rows = sweep_tau(
        model, val_loader, device, taus, k=cfg.k, nms_radius=cfg.nms_radius,
        output_stride=cfg.output_stride, match_radius_px=cfg.match_radius_px,
        class_names=cfg.class_names,
    )

    best = min(rows, key=lambda r: r["count_mae"])
    per_class_tau = {
        name: min(rows, key=lambda r: r["per_class"][name]["count_mae"])["tau"]
        for name in cfg.class_names
    }
    for name, t in per_class_tau.items():
        m = next(r for r in rows if r["tau"] == t)["per_class"][name]
        print(f"{name:>12}: best tau {t:.2f} -> MAE {m['count_mae']:.2f}, "
              f"F1 {m['f1']:.3f}, bias {m['count_bias']:+.2f}")

    fig, ax1 = plt.subplots(figsize=(8.5, 4.5))
    ax1.plot([r["tau"] for r in rows], [r["count_mae"] for r in rows], "o-", color="#c0392b")
    ax1.set_xlabel("tau"); ax1.set_ylabel("count MAE (macro)", color="#c0392b")
    ax2 = ax1.twinx()
    ax2.plot([r["tau"] for r in rows], [r["f1"] for r in rows], "s-", color="#2b5f9e",
             label="F1 (macro)")
    if cfg.n_classes > 1:
        for c, name in enumerate(cfg.class_names):
            ax2.plot([r["tau"] for r in rows], [r["per_class"][name]["f1"] for r in rows],
                     "--", color=class_colour(c), lw=1, label=f"F1 {name}")
        ax2.legend(loc="lower center", fontsize=8)
    ax2.set_ylabel("localization F1", color="#2b5f9e")
    ax1.axvline(best["tau"], color="gray", ls="--", lw=1)
    ax1.set_title(f"best tau {best['tau']:.2f}: MAE {best['count_mae']:.2f}, "
                f"F1 {best['f1']:.3f}, bias {best['count_bias']:+.2f}")
    return best, per_class_tau, fig


best, per_class_tau, fig = calibrate_tau(cfg, model, val_ds)
per_class_tau